In [20]:
import torch
import torchvision

from torch import nn
from torchvision import datasets, transforms
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [18]:
train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

In [33]:
train_transform = transforms.Compose([
    transforms.Resize(size=[64,64]),
    transforms.RandomHorizontalFlip(p = 1),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize(size=[64,64]),
    transforms.ToTensor()
])

#pic = Image.open("data/pizza.jpeg")
#pic = train_transform(pic)
#pic.show()

In [36]:
train_dataset = datasets.ImageFolder(root = train_dir, transform = train_transform)
test_dataset = datasets.ImageFolder(root = test_dir, transform = test_transform)

In [37]:
train_dataloader = DataLoader(
    dataset = train_dataset,
    batch_size = 32,
    shuffle = True,
    num_workers = 4,
    pin_memory = True
)

test_dataloader = DataLoader(
    dataset = test_dataset,
    batch_size = 32,
    shuffle = False,
    num_workers = 4,
    pin_memory = True
)

In [105]:
class CNN(nn.Module):

    def __init__(self, in_channel, hidden_unit, out_channel):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(
                in_channels = in_channel,
                out_channels = hidden_unit,
                kernel_size = 3,
                stride = 1,
                padding = 1,
            ),
            nn.BatchNorm2d(hidden_unit),
            nn.ReLU(),
            nn.Conv2d(
                in_channels = hidden_unit,
                out_channels = hidden_unit,
                kernel_size = 3,
                stride = 1,
                padding = 1
            ),
            nn.BatchNorm2d(hidden_unit),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size = 3,
                stride = 2
            )
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(
                in_channels = hidden_unit,
                out_channels = hidden_unit*2,
                kernel_size = 3,
                stride = 1,
                padding = 1,
            ),
            nn.BatchNorm2d(hidden_unit*2),
            nn.ReLU(),
            nn.Conv2d(
                in_channels = hidden_unit*2,
                out_channels = hidden_unit*2,
                kernel_size = 3,
                stride = 1,
                padding = 1
            ),
            nn.BatchNorm2d(hidden_unit*2),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size = 3,
                stride = 2
            )
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(
                in_channels = hidden_unit*2,
                out_channels = hidden_unit*4,
                kernel_size = 3,
                stride = 1,
                padding = 1,
            ),
            nn.BatchNorm2d(hidden_unit*4),
            nn.ReLU(),
            nn.Conv2d(
                in_channels = hidden_unit*4,
                out_channels = hidden_unit*4,
                kernel_size = 3,
                stride = 1,
                padding = 1
            ),
            nn.BatchNorm2d(hidden_unit*4),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size = 3,
                stride = 2
            )
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(
                in_channels = hidden_unit*4,
                out_channels = hidden_unit*8,
                kernel_size = 3,
                stride = 1,
                padding = 1,
            ),
            nn.BatchNorm2d(hidden_unit*8),
            nn.ReLU(),
            nn.Conv2d(
                in_channels = hidden_unit*8,
                out_channels = hidden_unit*8,
                kernel_size = 3,
                stride = 1,
                padding = 1
            ),
            nn.BatchNorm2d(hidden_unit*8),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size = 3,
                stride = 2
            )
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(
                in_features = hidden_unit*8,
                out_features = out_channel
            )
        )

    def forward(self, x):

        x = self.classifier(self.conv4(self.conv3(self.conv2(self.conv1(x)))))
        return x

In [106]:
model = CNN(
    in_channel = 3,
    hidden_unit = 16,
    out_channel = 3
)

model.parameters()

<generator object Module.parameters at 0x77c76f3e1300>

In [68]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params = model.parameters(), lr = 0.0003)

In [56]:
def train_loop(dataloader: torch.utils.data.DataLoader, model: torch.nn.Module, loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer):

    model.train()

    train_loss = 0

    for batch, (X, y) in enumerate(dataloader):

        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_prob = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)

    train_loss /= len(dataloader)
    
    return train_loss

In [66]:
def test_loop(dataloader: torch.utils.data.DataLoader, model: torch.nn.Module, loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer):

    model.eval()

    test_loss = 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):

            test_pred = model(X)
            loss = loss_fn(test_pred, y)
            test_loss += loss

            test_prob = torch.argmax(torch.softmax(test_pred, dim=1), dim=1)

        test_loss /= len(dataloader)

    return test_loss

In [64]:
def training(train_dataloader: troch.utils.data.DataLoader, test_dataloader: torch.utils.data.DataLoader, model: torch.nn.Module, loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer, epoch: int):

    results = {
        "train_loss": [],
        "test_loss": []
    }

    for epochs in tqdm(range(epoch)):
        train_loss = train_loop(
            dataloader = train_dataloader,
            model = model,
            loss_fn = loss_fn,
            optimizer = optimizer
        )

        test_loss = test_loop(
            dataloader = test_dataloader,
            model = model,
            loss_fn = loss_fn,
            optimizer = optimizer
        )

        results["train_loss"].append(train_loss)
        results["test_loss"].append(test_loss)

        print(f"TRAIN LOSS:{train_loss} | TEST LOSS:{test_loss} | EPOCH:{epochs+1}")

    return results

In [107]:
training(
    train_dataloader = train_dataloader,
    test_dataloader = test_dataloader,
    model = model,
    loss_fn = loss_fn,
    optimizer = optimizer,
    epoch = 20
)

  0%|          | 0/20 [00:00<?, ?it/s]

TRAIN LOSS:1.1297097206115723 | TEST LOSS:1.0569761991500854 | EPOCH:1
TRAIN LOSS:1.156545877456665 | TEST LOSS:1.0373514890670776 | EPOCH:2
TRAIN LOSS:1.159727692604065 | TEST LOSS:1.0229415893554688 | EPOCH:3
TRAIN LOSS:1.1802737712860107 | TEST LOSS:1.0372037887573242 | EPOCH:4
TRAIN LOSS:1.1401779651641846 | TEST LOSS:1.0881214141845703 | EPOCH:5
TRAIN LOSS:1.1805145740509033 | TEST LOSS:1.0935560464859009 | EPOCH:6
TRAIN LOSS:1.153554081916809 | TEST LOSS:1.1081573963165283 | EPOCH:7
TRAIN LOSS:1.196067452430725 | TEST LOSS:1.1215816736221313 | EPOCH:8
TRAIN LOSS:1.2119290828704834 | TEST LOSS:1.1225956678390503 | EPOCH:9
TRAIN LOSS:1.1970938444137573 | TEST LOSS:1.1074366569519043 | EPOCH:10
TRAIN LOSS:1.1912089586257935 | TEST LOSS:1.1042976379394531 | EPOCH:11
TRAIN LOSS:1.2097043991088867 | TEST LOSS:1.1216424703598022 | EPOCH:12
TRAIN LOSS:1.1402084827423096 | TEST LOSS:1.126822829246521 | EPOCH:13
TRAIN LOSS:1.140596628189087 | TEST LOSS:1.1214195489883423 | EPOCH:14
TRAIN L

{'train_loss': [tensor(1.1297, grad_fn=<DivBackward0>),
  tensor(1.1565, grad_fn=<DivBackward0>),
  tensor(1.1597, grad_fn=<DivBackward0>),
  tensor(1.1803, grad_fn=<DivBackward0>),
  tensor(1.1402, grad_fn=<DivBackward0>),
  tensor(1.1805, grad_fn=<DivBackward0>),
  tensor(1.1536, grad_fn=<DivBackward0>),
  tensor(1.1961, grad_fn=<DivBackward0>),
  tensor(1.2119, grad_fn=<DivBackward0>),
  tensor(1.1971, grad_fn=<DivBackward0>),
  tensor(1.1912, grad_fn=<DivBackward0>),
  tensor(1.2097, grad_fn=<DivBackward0>),
  tensor(1.1402, grad_fn=<DivBackward0>),
  tensor(1.1406, grad_fn=<DivBackward0>),
  tensor(1.2133, grad_fn=<DivBackward0>),
  tensor(1.1591, grad_fn=<DivBackward0>),
  tensor(1.1475, grad_fn=<DivBackward0>),
  tensor(1.2055, grad_fn=<DivBackward0>),
  tensor(1.1726, grad_fn=<DivBackward0>),
  tensor(1.1376, grad_fn=<DivBackward0>)],
 'test_loss': [tensor(1.0570),
  tensor(1.0374),
  tensor(1.0229),
  tensor(1.0372),
  tensor(1.0881),
  tensor(1.0936),
  tensor(1.1082),
  tens

In [ ]:
img = Image.open("data/3837522.jpg")
img = test_transform(img).unsqueeze(0)

choice = ["pizza", "steak", "sushi"]

model.eval()

with torch.no_grad():
    prob = model(img)
    classes = torch.argmax(torch.softmax(prob, dim=1), dim=1).item()

0